# First Pytorch Example (Fashion MNIST)

This Notebook is about detecting T-Shirts in Fashion MNIST. We limit it to T-Shirts to start with binary classification instead of multinomial classification.

Get the necessary files for the next session from https://www.kaggle.com/datasets/zalando-research/fashionmnist


## Preparing the data

In [ ]:
import pandas as pd
import numpy as np
import torch  
print(torch.__version__)

In [ ]:
#read data with pandas
dataTrain = pd.read_csv('data/fashion-mnist_train.csv')
dataTest = pd.read_csv('data/fashion-mnist_test.csv')
print("All read")

In [ ]:
# Look at the data, what can we see ? 
display(dataTrain)
#type(dataTrain)

In [ ]:
#interger location based indexing
X = dataTrain.iloc[:, 1:].to_numpy() # cut of the label column
y = dataTrain.iloc[:, 0].to_numpy() 

#Look at the data
print("X Shape", X.shape)
print("y Shape", y.shape)


In [ ]:
#convert to numpy
X_train, y_train = dataTrain.iloc[:, 1:].to_numpy(), dataTrain.iloc[:, 0].to_numpy()
X_test, y_test = dataTest.iloc[:, 1:].to_numpy(), dataTest.iloc[:, 0].to_numpy()

print("Training-data", X_train.shape)
print("Test-data", X_test.shape)

In [ ]:
#Define Classnames (taken from docu kaagle website)
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
class_names[3]

### Plot some examples with plt.imshow()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([]) 
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_train[i].reshape(28, 28), cmap="binary") # 
    plt.xlabel(class_names[y_train[i]])
plt.show()

### Relabel images to binary decision about t-shirt 

In [ ]:
# get from classes to binary classification for t-shirts 
y_train = y_train == 0
print("Nonzero", np.count_nonzero(y_train)) # 6000 t-Shirts 10%
y_test = y_test == 0
print("Nonzero", np.count_nonzero(y_test)) # 1000 t-Shirts 10%



In [ ]:
y_train

# Now classify all T-Shirts

In [ ]:
# Import Pytorch Layer Types 
import torch
import torch.nn as nn
import torch.optim as optim
print("Done")

In [ ]:
# Create a neuronal network with pytorch
# Define the model
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.inputlayer = nn.Linear(784, 100)
        self.hidden_activation = nn.Sigmoid()
        self.outputlayer = nn.Linear(100, 1)
        self.output_activation = nn.Sigmoid()  # for binary classification

    def forward(self, x):
        x = self.hidden_activation(self.inputlayer(x))
        x = self.output_activation(self.outputlayer(x))
        return x

model = SimpleNN()
print(model)

# Define loss function and optimizer
loss_fn = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
print(torch.version.cuda)
print(torch.__version__)
# Check if CUDA (GPU support) is available
if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available.")
#model.cuda()

In [ ]:
# Create DataLoader
from torch.utils.data import DataLoader, TensorDataset
X_tensor = torch.tensor(X_train, dtype=torch.float32)  # model input → float
y_tensor = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=1000, shuffle=True)

In [ ]:
# this is dooing the stuff that we know from the fit function
# Vanilla Training loop
for epoch in range(10):
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_y in loader:       
        optimizer.zero_grad() # reset old gradients
        outputs = model(batch_X) # Forward propagation

        loss = loss_fn(outputs, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_X.size(0)

        # Accuracy calculation
        predicted = (outputs >= 0.5).float()
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    print(f"Epoch {epoch+1}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.4f}")

# Calculate training and test accuracy

In [ ]:
from torchmetrics.classification import BinaryAccuracy
metric = BinaryAccuracy()

acc = metric(model(X_tensor), y_tensor)
print("Training Accuracy", acc)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)  # model input → float
y_test_tensor = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)
acc = metric(model(X_test_tensor), y_test_tensor)
print("Test Accuracy", acc)
